In [ ]:
#| default_exp encoder

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, inspect, numpy as np
from pathlib import Path
from fastcore.all import ifnone, patch
from laghu.core import log, resolve, model_files, st_map
from laghu.unigram import tokenizer, Unigram

## LeanModel

`LeanModel` is laghu's replacement for model2vec's `StaticModel`. It has the parts of `StaticModel` that use a model, as opposed to training or saving one: the methods `tokenize`, `encode` and `encode_as_sequence`, and the attributes `embedding`, `weights`, `token_mapping`, `normalize`, `config`, `dim`, `median_token_length`, `unk_token_id` and `tokens`. Code written for a `StaticModel` that only encodes text works unchanged.

Encoding uses the same numpy operations as model2vec, on the same batch boundaries, so even the cases where floating-point details matter come out identical: a float16 table, and the float64 zero row model2vec produces for an empty text. What differs is where things are stored:

- `embedding` is a read-only `np.memmap` of `model.safetensors`, and so are `weights` and `token_mapping` when a vocabulary-quantized model has them.
- The tokenizer is laghu's `Unigram` for a Unigram model, with its vocabulary mapped from a shared file (`shared=False` keeps it in a dict), and `tokenizers` for anything else.
- `tokens`, the list of every token in id order, is only built the first time you read it, since for 500,000 tokens it takes about 40 MB.

`LeanModel.from_pretrained(path)` takes a local folder or a Hugging Face repo id, as `StaticModel.from_pretrained` does; a repo already in the cache loads offline. `normalize` overrides the model's setting for scaling output vectors to unit length, and `lean=False` makes it use `tokenizers` even for a Unigram model. Usually you call `load` instead, below. `lean` reports whether laghu's own tokenizer is in use. `encode([])` returns an empty `(0, dim)` array where model2vec raises, and `use_multiprocessing` is accepted and ignored.

In [ ]:
#| export
class LeanModel:
    "A stand-in for model2vec's `StaticModel` that maps its table, and a Unigram vocabulary, from files instead of loading them."
    def __init__(self, folder, normalize=None, lean=True, shared=True):
        f = model_files(folder); self.folder = Path(folder)
        self.config = json.loads(f.cfg.read_text())
        self.embedding, self.weights, self.token_mapping = st_map(f.emb, [f.name, 'weights', 'mapping'])
        if self.embedding is None: raise KeyError(f'{f.emb}: no tensor {f.name!r}')
        self.tokenizer = tokenizer(f.tok, lean, shared)
        if self.token_mapping is None and self.tokenizer.n != len(self.embedding):
            raise ValueError(f'Number of tokens ({self.tokenizer.n}) does not match number of vectors ({len(self.embedding)}).')
        self.median_token_length, self.unk_token_id = self.tokenizer.median, self.tokenizer.unk_token_id
        self.normalize = ifnone(normalize, self.config.get('normalize', False))
        self.config['normalize'] = self.normalize

    @classmethod
    def from_pretrained(cls, path, token=None, normalize=None, subfolder=None, force_download=False, lean=True, shared=True):
        "A `LeanModel` for a folder or Hugging Face repo id `path`, loading a cached repo offline."
        return cls(resolve(path, token=token, subfolder=subfolder, force_download=force_download), normalize=normalize, lean=lean, shared=shared)

    @property
    def dim(self): return self.embedding.shape[1]
    @property
    def embedding_dtype(self): return np.dtype(self.embedding.dtype).name
    @property
    def lean(self): return isinstance(self.tokenizer, Unigram)
    @property
    def tokens(self):
        "Every token in id order, as `StaticModel.tokens` gives them."
        if not hasattr(self, '_tokens'): v = self.tokenizer.vocab(); self._tokens = tuple(sorted(v, key=v.get))
        return self._tokens

    def __repr__(self):
        return f"{type(self).__name__}({str(self.folder)!r}, {len(self.embedding)}×{self.dim} {self.embedding_dtype}, {'unigram' if self.lean else 'tokenizers'})"

`tokenize(sentences, max_length)` returns the token ids of each text, following model2vec's steps in the same order. With `max_length` set, it first cuts each text to `max_length × median_token_length` characters, which is enough characters to fill `max_length` tokens on average. It then tokenizes, removes the unknown token's id where the model has one (WordPiece and WordLevel models do, Unigram models do not), and keeps the first `max_length` ids.

`encode(sentences)` gives each text's vector: the mean of its tokens' vectors, scaled to unit length when `normalize` is on. A single string gives a single vector, and a list gives an array with one row per text. Texts are processed `batch_size` (1,024) at a time and cut to `max_length` (512) tokens. `encode_as_sequence(sentences)` gives each text's token vectors without averaging, as an array of shape `(n_tokens, dim)`. In a vocabulary-quantized model each token's row is found through `token_mapping`, and in a model with `weights` each token's vector is multiplied by its weight.

When the tokenizer has a Rust path (`Unigram.rs`, see [unigram](unigram.html)), `tokenize` uses it, and for a float32 table without `weights` or `token_mapping` `encode` also averages in Rust, reading the mapped table in place. The averaging is numpy's `mean(axis=0)` step for step in float32: the first row copied, the others added in order, the sum divided by the count. A text with no tokens leaves a zero row and turns the batch into float64, as model2vec's float64 zero row does, and the unit scaling stays numpy's. Every other model averages with numpy as before.

In [ ]:
#| export
def _batches(xs, bs): return (xs[i:i + bs] for i in range(0, len(xs), bs))

@patch
def _cut(self:LeanModel, sentences, max_length):
    "`sentences` cut to `max_length` times the median token length in characters, as model2vec cuts them before tokenizing."
    if max_length is None: return sentences
    m = max_length * self.median_token_length; return [s[:m] for s in sentences]

@patch
def tokenize(self:LeanModel, sentences, max_length=None):
    "The token ids of each text in `sentences`, as `StaticModel.tokenize` gives them."
    sentences = self._cut(sentences, max_length)
    if (rs := getattr(u := self.tokenizer, 'rs', None)) is not None:
        xs, pre = u.rs_texts(sentences); return rs.batch(xs, self.unk_token_id, max_length, u.threads, pre)
    ids = self.tokenizer(sentences)
    if self.unk_token_id is not None: ids = [[k for k in o if k != self.unk_token_id] for o in ids]
    return [o[:max_length] for o in ids] if max_length is not None else ids

@patch
def _rows(self:LeanModel, ids):
    "The vectors of token `ids`, through `token_mapping` and scaled by `weights` when the model has them."
    emb = self.embedding[ids if self.token_mapping is None else self.token_mapping[ids]]
    return emb * self.weights[ids][:, None] if self.weights is not None else emb

@patch
def _pool_rs(self:LeanModel):
    "Whether `laghu_rs` can both tokenize and pool: a Unigram on the Rust path, and a C-contiguous float32 table without weights or a token mapping."
    e = self.embedding
    return getattr(self.tokenizer, 'rs', None) is not None and self.weights is None and self.token_mapping is None and e.dtype == np.float32 and e.flags.c_contiguous

@patch
def _encode_batch(self:LeanModel, sentences, max_length):
    "Encode one batch as model2vec does, where an empty text's float64 zero row makes the batch float64."
    if self._pool_rs():
        u, sentences = self.tokenizer, self._cut(sentences, max_length)
        out = np.empty((len(sentences), self.dim), np.float32)
        xs, pre = u.rs_texts(sentences)
        n = u.rs.encode(xs, self.embedding, out, self.unk_token_id, max_length, u.threads, pre)
        if 0 in n: out = out.astype(np.float64)
    else: out = np.stack([self._rows(o).mean(axis=0) if o else np.zeros(self.dim) for o in self.tokenize(sentences, max_length)])
    return out / (np.linalg.norm(out, axis=1, keepdims=True) + 1e-32) if self.normalize else out

@patch
def encode(self:LeanModel, sentences, show_progress_bar=False, max_length=512, batch_size=1024, use_multiprocessing=True, multiprocessing_threshold=10_000, **kwargs):
    "The mean token vector of each text, unit length when `normalize` is set, as `StaticModel.encode` gives it."
    one = isinstance(sentences, str)
    xs = [sentences] if one else list(sentences)
    if not xs: return np.zeros((0, self.dim), self.embedding.dtype)
    out = np.concatenate([self._encode_batch(b, max_length) for b in _batches(xs, batch_size)], axis=0)
    return out[0] if one else out

@patch
def encode_as_sequence(self:LeanModel, sentences, max_length=None, batch_size=1024, show_progress_bar=False, use_multiprocessing=True, multiprocessing_threshold=10_000):
    "The token vectors of each text, shape `(n_tokens, dim)`, as `StaticModel.encode_as_sequence` gives them."
    one = isinstance(sentences, str)
    xs = [sentences] if one else list(sentences)
    out = [self._rows(o) if o else np.zeros((0, self.dim)) for b in _batches(xs, batch_size) for o in self.tokenize(b, max_length)]
    return out[0] if one else out

## load

`load(path)` is the usual way in. It returns a `LeanModel` for a folder or Hugging Face repo id. If laghu cannot load the model, it logs a warning saying why and tries model2vec's `StaticModel`, so a model laghu does not support still loads. If that fails too, it raises laghu's original error. `fallback=False` raises at once instead. Only the fallback imports model2vec, which needs the `laghu[fallback]` extra. The other arguments are passed to `LeanModel.from_pretrained`.

In [ ]:
#| export
def load(path, normalize=None, token=None, subfolder=None, force_download=False, lean=True, shared=True, fallback=True):
    "Load `path` as a `LeanModel`, falling back to model2vec's `StaticModel` with a warning unless `fallback=False`."
    try: return LeanModel.from_pretrained(path, token=token, normalize=normalize, subfolder=subfolder, force_download=force_download, lean=lean, shared=shared)
    except Exception as e:
        if not fallback: raise
        log.warning(f'laghu: cannot read {path} ({e!r}); loading it with model2vec instead')
        try:
            from model2vec import StaticModel
            return StaticModel.from_pretrained(path, token=token, normalize=normalize, subfolder=subfolder, force_download=force_download)
        except Exception as e2: raise e from e2

## Tests

`same_as_model2vec(path, txts)` loads a model with both laghu and model2vec and compares them. It checks `tokenize` and `encode` on every text, comparing vectors as float32, as float16 and bit for bit, and `encode_as_sequence` on the first `seq` (200) texts. It returns a dict with one True or False per check, plus `lean` and the number of texts `n`; the index page lists the keys. laghu exports it so you can check a model of your own. Without `txts` it uses `corpus()`.

In [ ]:
#| export
def same_as_model2vec(path, txts=None, max_length=512, seq=200):
    "Compare `load(path)` with model2vec on `txts`, default `corpus()`, returning a dict of the checks."
    from model2vec import StaticModel
    from laghu.core import corpus
    if 'max_length' not in inspect.signature(StaticModel.tokenize).parameters:
        raise ImportError('same_as_model2vec needs model2vec 0.9, whose `StaticModel.tokenize` takes `max_length`: install `laghu[fallback]`')
    txts = ifnone(txts, corpus())
    lm, sm = load(path, fallback=False), StaticModel.from_pretrained(str(resolve(path)), force_download=False)
    a, b = lm.encode(txts, max_length=max_length), sm.encode(txts, max_length=max_length)
    sa, sb = lm.encode_as_sequence(txts[:seq]), sm.encode_as_sequence(txts[:seq])
    return dict(lean=lm.lean, n=len(txts), ids=lm.tokenize(txts, max_length) == sm.tokenize(txts, max_length), dtype=a.dtype == b.dtype,
        f32=np.array_equal(a.astype(np.float32), b.astype(np.float32)), f16=np.array_equal(a.astype(np.float16), b.astype(np.float16)),
        exact=a.dtype == b.dtype and np.array_equal(a, b), seq=all(x.dtype == y.dtype and np.array_equal(x, y) for x, y in zip(sa, sb)),
        median=lm.median_token_length == sm.median_token_length, unk=lm.unk_token_id == sm.unk_token_id, tokens=lm.tokens == tuple(sm.tokens))

In [ ]:
import sys, types, tomllib
from unittest import mock

In [ ]:
class M2V010:
    "model2vec 0.10's `StaticModel`, whose `tokenize` takes no `max_length`."
    def tokenize(self, sentences): return []
with mock.patch.dict(sys.modules, model2vec=types.SimpleNamespace(StaticModel=M2V010)):
    try: same_as_model2vec('.'); raise AssertionError('compared with model2vec 0.10')
    except ImportError as e: assert 'max_length' in str(e), e
fb = tomllib.loads(Path('../pyproject.toml').read_text())['project']['optional-dependencies']['fallback']
assert any('<0.10' in o for o in fb), fb

In [ ]:
import importlib.util, os
from laghu.core import try_resolve, corpus, ODD
M2V = importlib.util.find_spec('model2vec') is not None   # the `fallback` extra; the tests that compare with it skip without it
RS = importlib.util.find_spec('laghu_rs') is not None     # the optional Rust core; the tests of its path skip without it
txts = corpus(1500)
ok = lambda r: all(v for k, v in r.items() if k not in ('lean', 'n'))

In [ ]:
if M2V and try_resolve('minishlab/potion-multilingual-128M'):
    m = load('minishlab/potion-multilingual-128M')
    assert isinstance(m, LeanModel) and m.lean and isinstance(m.embedding, np.memmap) and not m.embedding.flags.writeable and m.tokenizer.mapped is not None
    r = same_as_model2vec('minishlab/potion-multilingual-128M', txts); print(r); assert ok(r) and r['lean']
    for ml in (None, 8): assert m.tokenize(ODD, ml) == __import__('model2vec').StaticModel.from_pretrained(str(m.folder), force_download=False).tokenize(ODD, ml)

With `laghu_rs` installed, potion-multilingual-128M tokenizes and averages in Rust, from normalizing to the mean. The Rust path, the Rust path segmenting what Python's `split` gives (`tokenizer.rs_pre` set to None), the Python path (`LAGHU_RUST=0`) and model2vec give the same ids and the same vectors, dtype and bits, on all of `corpus(3000)`, at `max_length` 512 and None, both for the texts that have tokens alone, whose batches stay float32, and with an empty text among them, which makes its batch float64:

In [ ]:
def python_path(path):
    "`load(path)` made with `LAGHU_RUST=0`, so it tokenizes and averages in Python."
    os.environ['LAGHU_RUST'] = '0'
    try: return load(path, fallback=False)
    finally: del os.environ['LAGHU_RUST']

if RS and M2V and (f := try_resolve('minishlab/potion-multilingual-128M')):
    big, sm = corpus(3000), __import__('model2vec').StaticModel.from_pretrained(str(f), force_download=False)
    r, q, p = load(f, fallback=False), load(f, fallback=False), python_path(f)
    q.tokenizer.rs_pre = None   # Rust segments what Python's `split` gives
    assert r.tokenizer is not q.tokenizer and r._pool_rs() and r.tokenizer.rs_pre is not None and q._pool_rs() and p.tokenizer.rs is None
    seen = {}
    for ml in (512, None):
        ids = sm.tokenize(big, ml); assert all(m.tokenize(big, ml) == ids for m in (r, q, p))
        full = [s for s, o in zip(big, ids) if o]
        for xs in (full, full[:700] + [''] + full[700:]):
            e = sm.encode(xs, max_length=ml); seen[ml, len(xs)] = e.dtype
            assert all(o.dtype == e.dtype and np.array_equal(o, e) for o in (m.encode(xs, max_length=ml) for m in (r, q, p)))
    seen

In [ ]:
if M2V and try_resolve('minishlab/potion-base-8M'):   # WordPiece: tokenizers tokenizes, the table is still mapped
    r = same_as_model2vec('minishlab/potion-base-8M', txts); print(r); assert ok(r) and not r['lean']

In [ ]:
if M2V and try_resolve('minishlab/potion-code-16M-v2'):   # a float16 table
    m = load('minishlab/potion-code-16M-v2'); assert m.embedding_dtype == 'float16'
    r = same_as_model2vec('minishlab/potion-code-16M-v2', txts); print(r); assert ok(r)

The edge cases: a single string, an empty list, `normalize` turned off, the float64 row of an empty text, and the fallback to model2vec, both when it saves a load and when model2vec fails too.

In [ ]:
if try_resolve('minishlab/potion-base-8M'):
    m = load('minishlab/potion-base-8M')
    assert m.encode('hello').shape == (m.dim,) and m.encode([]).shape == (0, m.dim)
    assert m.encode(['', 'hello']).dtype == np.float64 and m.encode(['hello']).dtype == np.float32   # as model2vec: the zero row is float64
    assert not np.allclose(np.linalg.norm(load('minishlab/potion-base-8M', normalize=False).encode(['hello world'])), 1)
    assert m.encode_as_sequence('') .shape == (0, m.dim) and m.encode_as_sequence('hello world').shape[1] == m.dim

In [ ]:
import logging, tempfile, shutil
from laghu.core import st_header
from fastcore.all import Path
if f := try_resolve('minishlab/potion-base-8M'):
    bf = Path(tempfile.mkdtemp())
    for o in ('config.json', 'tokenizer.json'): shutil.copy(f/o, bf/o)
    hdr, at = st_header(f/'model.safetensors'); hdr['embeddings']['dtype'] = 'BF16'; hdr['embeddings']['shape'][1] *= 2
    h = json.dumps(hdr).encode(); h += b' ' * (-len(h) % 8)
    (bf/'model.safetensors').write_bytes(len(h).to_bytes(8, 'little') + h + (f/'model.safetensors').read_bytes()[at:])
    try: load(bf); raise AssertionError('loaded a BF16 table')
    except Exception as e: assert 'BF16' in str(e), e        # model2vec fails too, so laghu's error is the one raised
    shutil.copy(f/'model.safetensors', bf/'model.safetensors')
    (bf/'config.json').write_text('{not json')            # laghu cannot read the config; model2vec cannot either
    try: load(bf); raise AssertionError('loaded a broken config')
    except json.JSONDecodeError: pass

In [ ]:
if M2V and try_resolve('minishlab/potion-base-8M'):
    from unittest import mock
    from model2vec import StaticModel
    with mock.patch.object(LeanModel, '__init__', side_effect=RuntimeError('boom')):
        logging.getLogger('laghu').propagate = True
        m = load('minishlab/potion-base-8M')
    assert isinstance(m, StaticModel)

Two models are built here from parts, to reach code that no public potion model uses. One is vocabulary-quantized: its table has 1,000 rows, with a `mapping` from each token to one of them and a `weights` value per token. The other puts t5-small's tokenizer (no model type, WhitespaceSplit before Metaspace, 103 added tokens) on a random float16 table.

In [ ]:
from laghu.core import st_write
rng = np.random.default_rng(0)
if M2V and (f := try_resolve('minishlab/potion-base-8M')):
    vq = Path(tempfile.mkdtemp()); shutil.copy(f/'tokenizer.json', vq/'tokenizer.json')
    emb, = st_map(f/'model.safetensors', ['embeddings'])
    st_write(vq/'model.safetensors', dict(embeddings=np.array(emb[:1000]), mapping=rng.integers(0, 1000, len(emb)).astype(np.int64),
                                          weights=rng.random(len(emb)).astype(np.float32)))
    (vq/'config.json').write_text(json.dumps(dict(normalize=True, vocabulary_quantization=1000)))
    m = load(vq); assert m.token_mapping is not None and m.weights is not None
    r = same_as_model2vec(vq, txts); print(r); assert ok(r)

In [ ]:
def hub_file(repo, fn):
    from huggingface_hub import hf_hub_download
    try: return Path(hf_hub_download(repo, fn))
    except Exception as e: print(f'skipping {repo}: {e!r}')

if M2V and (fn := hub_file('google-t5/t5-small', 'tokenizer.json')):
    t5 = Path(tempfile.mkdtemp()); shutil.copy(fn, t5/'tokenizer.json')
    st_write(t5/'model.safetensors', dict(embeddings=rng.standard_normal((32100, 64)).astype(np.float16)))
    (t5/'config.json').write_text(json.dumps(dict(normalize=False)))
    m = load(t5); assert m.lean and m.embedding_dtype == 'float16'
    r = same_as_model2vec(t5, txts); print(r); assert ok(r)